# Day 1 - 배터리 데이터 로드

`archive/`의 `.mat` 파일(MATLAB v7.3 = HDF5)을 `h5py`로 읽습니다.

- `summary`: 셀별 cycle 단위 요약값 (IR, QDischarge, Tavg, chargetime 등) → 가볍고 빠름
- `cycles`: cycle 내부 시계열 (V, I, T, Qd, Qdlin 등) → 무거우므로 필요한 셀만 따로 로드

In [5]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

DATA_DIR = Path("archive")
mat_files = sorted(DATA_DIR.glob("*.mat"))
for p in mat_files:
    print(f"{p.name}  ({p.stat().st_size / 1e9:.2f} GB)")

2017-05-12_batchdata_updated_struct_errorcorrect.mat  (3.03 GB)
2018-02-20_batchdata_updated_struct_errorcorrect.mat  (2.02 GB)
2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat  (0.09 GB)
2018-04-12_batchdata_updated_struct_errorcorrect.mat  (3.24 GB)


In [6]:
SUMMARY_KEYS = ["cycle", "QDischarge", "QCharge", "IR", "Tavg", "Tmin", "Tmax", "chargetime"]
CYCLE_KEYS = ["t", "V", "I", "T", "Qc", "Qd", "Qdlin", "Tdlin", "discharge_dQdV"]


def _read_str(f, ref):
    d = f[ref]
    # MATLAB 'string' 객체(barcode, channel_id)는 h5py로 디코딩 불가 → None
    if d.attrs.get("MATLAB_class") != b"char":
        return None
    return "".join(chr(c) for c in d[()].ravel())


def _read_num(f, ref):
    arr = np.asarray(f[ref][()]).ravel()
    return float(arr[0]) if arr.size else np.nan


def load_batch(path):
    """한 batch 파일에서 셀별 메타데이터 + summary(DataFrame)를 읽는다."""
    path = Path(path)
    batch_name = path.name.split("_batchdata")[0]
    cells = []
    with h5py.File(path, "r") as f:
        b = f["batch"]
        for i in range(b["summary"].shape[0]):
            s = f[b["summary"][i, 0]]
            summary = pd.DataFrame({k: s[k][()].ravel() for k in SUMMARY_KEYS})
            cells.append({
                "batch": batch_name,
                "cell_idx": i,
                "cell_id": f"{batch_name}_c{i}",
                "barcode": _read_str(f, b["barcode"][i, 0]),
                "channel_id": _read_str(f, b["channel_id"][i, 0]),
                "policy": _read_str(f, b["policy_readable"][i, 0]),
                "cycle_life": _read_num(f, b["cycle_life"][i, 0]),
                "summary": summary,
            })
    return cells


def load_cycles(path, cell_idx, cycle_range=None):
    """특정 셀의 cycle 내부 시계열을 읽는다. 반환: {cycle_no: {key: np.ndarray}}"""
    with h5py.File(path, "r") as f:
        c = f[f["batch"]["cycles"][cell_idx, 0]]
        n = c["I"].shape[0]
        idxs = range(n) if cycle_range is None else [j for j in cycle_range if j < n]
        return {
            j: {k: f[c[k][j, 0]][()].ravel() for k in CYCLE_KEYS}
            for j in idxs
        }

## 셀 메타데이터 / summary DataFrame

가장 작은 varcharge 파일(0.09GB)로 먼저 확인. 다른 batch는 `mat_files[i]`로 바꾸면 됨 (큰 파일은 수십 초 걸림).

In [7]:
cells = load_batch(mat_files[2])

meta_df = pd.DataFrame([{k: v for k, v in c.items() if k != "summary"} for c in cells])
print(meta_df.shape)
meta_df.head()

(2, 7)


,batch,cell_idx,cell_id,barcode,channel_id,policy,cycle_life
0,2018-04-03_varcharge,0,2018-04-03_varcharge_c0,None,None,VarCharge-2C-100cycles,NaN
1,2018-04-03_varcharge,1,2018-04-03_varcharge_c1,None,None,VarCharge-4C-100cycles,NaN


In [8]:
summary_df = pd.concat(
    [c["summary"].assign(cell_id=c["cell_id"]) for c in cells], ignore_index=True
)
print(summary_df.shape)
summary_df.head()

(1053, 9)


,cycle,QDischarge,QCharge,IR,Tavg,Tmin,Tmax,chargetime,cell_id
0,1.0,1.047808,1.047330,0.0,29.587036,25.356219,33.533627,24.212377,2018-04-03_varcharge_c0
1,2.0,1.052599,1.051810,0.0,34.521980,28.519470,40.943689,24.174765,2018-04-03_varcharge_c0
2,3.0,1.053151,1.053001,0.0,35.733892,32.620396,39.202064,24.199827,2018-04-03_varcharge_c0
3,4.0,1.053068,1.053090,0.0,35.573794,32.195839,39.510422,24.244183,2018-04-03_varcharge_c0
4,5.0,1.053388,1.053357,0.0,35.552473,28.806255,40.256791,24.222657,2018-04-03_varcharge_c0


## cycle 내부 시계열 DataFrame (셀 1개, cycle 몇 개만)

In [9]:
cyc = load_cycles(mat_files[2], cell_idx=0, cycle_range=range(1, 4))

# 길이가 같은 원시 시계열만 (Qdlin/Tdlin/discharge_dQdV는 1000포인트 보간값이라 길이가 다름)
RAW_KEYS = ["t", "V", "I", "T", "Qc", "Qd"]
cycle_df = pd.concat(
    [pd.DataFrame({k: v[k] for k in RAW_KEYS}).assign(cycle=j) for j, v in cyc.items()],
    ignore_index=True,
)
print(cycle_df.shape)
cycle_df.head()

(3894, 7)


,t,V,I,T,Qc,Qd,cycle
0,0.000000,2.320943,0.219882,30.751707,0.000002,6.536880e-07,1
1,0.000570,2.331793,0.339878,30.751112,0.000002,6.536880e-07,1
2,0.001015,2.343177,0.439973,30.750648,0.000002,6.536880e-07,1
3,0.001333,2.353343,0.520018,30.750316,0.000002,6.536880e-07,1
4,0.001868,2.367341,0.620086,30.749758,0.000002,6.536880e-07,1
